# Olivine (I13 scan 198065) – parallel-beam reconstruction with CIL

This notebook loads the raw Diamond I13 data, sets up a CIL parallel-beam geometry and reconstructs with CIL.
All tomography steps use CIL. Only metadata reading uses plain `h5py`/`numpy`.

**Pipeline:** bad-pixel masking, flat/dark normalisation, flux normalisation with the ion chamber and -log, then (each shown without and with)
1. ROI padding (`Padder`),
2. ring removal (`RingRemover`),
3. centre of rotation (`CentreOfRotationCorrector` estimate plus visual sweeps),

then FBP, and finally iterative reconstruction of one slice: SIRT with non-negativity, and FISTA with TV and non-negativity.

**Data**
- `198065.nxs`: scan metadata (`image_key`, rotation angles `ss1_theta`).
- `hama-198065.hdf`: raw frames at `/entry/data/data`, shape (4081, 1744, 1700), uint16. That's 40 darks, 40 flats and 4001 projections over 0–180°.
  The `.nxs` external links to the frames point to a path that doesn't exist, so the `.hdf` is read directly.

**Assumptions**
- Frame axes are (angle, vertical, horizontal), with the rotation axis vertical.
- The pixel size (`PIXEL_SIZE_UM`) isn't stored in the file. The estimate from the optics is 0.65 µm (6.5 µm pco.edge / 10× objective). The reconstructed values are in 1/µm.

**Things to know about this dataset**
- The sample is a slab wider than the field of view (ROI / interior tomography). Without padding, this causes cupping and a bright rim at the edge of the reconstruction circle.
- The flat field has concentric structure, and the flats don't fully match the beam later in the scan, which causes ring artefacts.
- Contrast is very low, so CIL's `image_sharpness` centre-of-rotation method doesn't converge. Instead, this notebook uses `xcorrelation` for the estimate,
  plus visual sweeps of FBP slices so you can check or override it (`COR_OFFSET_PX`).

In [ ]:
import os, time
import h5py
import numpy as np
import matplotlib.pyplot as plt

from cil.io.utilities import HDF5_utilities
from cil.framework import AcquisitionGeometry
from cil.processors import Normaliser, TransmissionAbsorptionConverter, CentreOfRotationCorrector, Binner, RingRemover, Padder, FluxNormaliser, MaskGenerator, Masker
from cil.recon import FBP
from cil.utilities.display import show2D, show_geometry
from scipy.ndimage import median_filter


def fov_mask(shape, frac=0.45):
    # disc covering the inner part of the reconstruction circle (excludes the bright truncation rim)
    ny, nx = shape[-2:]
    yy, xx = np.mgrid[:ny, :nx]
    return (yy - ny / 2) ** 2 + (xx - nx / 2) ** 2 < (frac * min(nx, ny)) ** 2


def fov_range(arr, q=(1, 99)):
    # display range from percentiles inside the field-of-view disc
    return tuple(np.percentile(arr[..., fov_mask(arr.shape)], q))

## Configuration
- `MODE='slices'` reconstructs the detector rows selected by `SLICE_ROWS`. The rows must be evenly spaced (a `slice` with a step), so they form a valid
  3D geometry with vertical pixel size `step × PIXEL_SIZE_UM`. Each selected row takes about 40 s to read from the share.
- `MODE='full'` reconstructs the whole volume. This reads 23 GB and needs about 47 GB of RAM for the float32 projections, plus about 20 GB for the volume at `FULL_BIN=1`.
  `FULL_BIN` bins the vertical and horizontal detector axes after normalisation.

In [ ]:
DATA_DIR = '/mnt/share/olivine_data/olivine data'
NXS = os.path.join(DATA_DIR, '198065.nxs')
HDF = os.path.join(DATA_DIR, 'hama-198065.hdf')
DSET = '/entry/data/data'

MODE = 'slices'                       # 'slices' or 'full'
SLICE_ROWS = slice(436, 1309, 436)    # detector rows 436, 872, 1308 (used in 'slices' mode)
FOCUS_ROW = 436                       # detector row used for the without/with comparisons, sweeps and iterative reconstruction
FULL_BIN = 1                          # detector binning in 'full' mode

PIXEL_SIZE_UM = 1                  # detector pixel size at the sample (µm) - TODO:
ION_NORMALISE = True                  # CIL FluxNormaliser with the ion-chamber readings
MASK_OUTLIERS = True                  # mask bad pixels in raw projections, flats and darks before normalisation (CIL MaskGenerator + Masker)
HOT_PIXEL_DARK_COUNTS = 1000          # hot pixels: mean dark above this (typical dark ~410-470)
OUTLIER_MAD = 5                       # persistent outliers: angle-averaged projections more than this x MAD from their horizontal neighbours
OUTLIER_WINDOW = 9                    # horizontal neighbourhood (pixels) for the persistent-outlier test
ZINGER_MASK = False                   # also mask one-off outliers (zingers) in individual projections; needs a 2nd copy of the data
ZINGER_MAD = 12                       # zinger threshold (x MAD) relative to a 5-projection angular median

# 1. ROI padding
PAD = True                            # pad the sinograms horizontally (ROI scan)
PAD_MODE = 'edge'                     # CIL Padder mode: 'edge', 'linear_ramp', 'constant', ...
PAD_WIDTH_PX = 400                    # padding on each side (detector pixels); also sets the iterative image size

# 3. centre of rotation
COR_OFFSET_PX = -15                  # final COR offset (detector pixels); None: use the CIL image_sharpness estimate (by eye, -20 to -10 looked best on the bubble row)
COR_SHARPNESS_SEARCH_PX = 40          # image_sharpness search range (pixels)
COR_SHARPNESS_TOL = 0.05              # image_sharpness tolerance
COR_COARSE_PX = np.arange(-40, 41, 5) # offsets for the coarse sweep (focus row)
COR_FINE_CENTRE_PX = -15           # centre of the fine sweep; None: image_sharpness estimate
COR_FINE_HALFWIDTH_PX = 6             # fine sweep covers centre ± this
COR_FINE_STEP_PX = 1
COR_ZOOM = (slice(700, 1300), slice(550, 1150))   # (y, x) region shown zoomed, unbinned pixels (centre + grain in row 436)

# 2. ring removal
RING_REMOVAL = True                   # apply CIL RingRemover to the data before reconstruction
RING_DECNUM = 6                       # wavelet decomposition levels (more = removes broader rings)
RING_WNAME = 'db10'
RING_SIGMA = 4                        # damping of the stripe filter (larger = stronger)
RING_SWEEP = [(4, 1.5), (4, 3), (6, 3), (6, 6), (8, 6)]   # (decNum, sigma) settings compared in the sweep

# iterative reconstruction (one slice, all angles)
ITER_BIN = 2                          # horizontal binning of the focus-row sinogram for the iterative methods (speed)
SIRT_ITERATIONS = 100
FISTA_ITERATIONS = 100
TV_ALPHA = 1.0                       # TV weight (depends on data scaling / PIXEL_SIZE_UM and ITER_BIN); 3 noisy, 10 patchy at ITER_BIN=2
TV_INNER_ITERATIONS = 10              # iterations of the TV proximal solver per FISTA step
TV_ALPHA_SWEEP = []                   # e.g. [0.3, 1, 3, 10] to compare TV weights

BACKEND = 'tigre'
FRAME_CHUNK = 200                     # frames per HDF5 read

## Metadata

In [ ]:
with h5py.File(NXS, 'r') as f:
    image_key = f['entry1/hama_sw_hdf/image_key'][()].astype(int)
    theta = f['entry1/hama_sw_hdf/ss1_theta'][()]
    ion = f['entry1/hama_sw_hdf/ionc_i'][()]        # ion chamber (beam intensity) per frame

meta = HDF5_utilities.get_dataset_metadata(HDF, DSET)
print(meta)
n_frames, n_vert, n_horiz = meta['shape']
assert n_frames == len(image_key)

def frame_range(key):
    idx = np.where(image_key == key)[0]
    assert np.all(np.diff(idx) == 1), 'frames of this type are not contiguous'
    return slice(int(idx[0]), int(idx[-1]) + 1)

dark_sel, flat_sel, proj_sel = frame_range(2), frame_range(1), frame_range(0)
theta_proj = theta[proj_sel]
n_proj = len(theta_proj)
print(f'darks {dark_sel}, flats {flat_sel}, projections {proj_sel}')
print(f'{n_proj} projections, {theta_proj.min():.3f} to {theta_proj.max():.3f} deg, mean step {np.diff(theta_proj).mean():.4f} deg')

## Look at the raw data

In [ ]:
row_sel = SLICE_ROWS if MODE == 'slices' else slice(None)
rows = range(n_vert)[row_sel]
row_step = row_sel.step or 1
assert len(rows) >= 2, 'select at least two rows'
print(f'{MODE} mode: {len(rows)} detector rows', list(rows) if MODE == 'slices' else '')
assert FOCUS_ROW in rows, 'FOCUS_ROW must be one of the selected detector rows'
v_focus = rows.index(FOCUS_ROW)

frame = lambda i: HDF5_utilities.read(HDF, DSET, source_sel=(slice(i, i + 1), slice(None), slice(None)))[0]
frames = [frame(dark_sel.start), frame(flat_sel.start), frame(proj_sel.start), frame(proj_sel.start + n_proj // 2)]
show2D(frames, title=['dark', 'flat', 'projection 0 deg', 'projection 90 deg'],
       fix_range=[tuple(np.percentile(f, [0.5, 99.5])) for f in frames],
       origin='upper-left', num_cols=4, size=(20, 5));

## Geometry
Parallel-beam 3D geometry in µm. In slice mode, the vertical pixel size is `row_step × PIXEL_SIZE_UM`.

In [ ]:
ag = (AcquisitionGeometry.create_Parallel3D(units='um')
      .set_panel((n_horiz, len(rows)), pixel_size=(PIXEL_SIZE_UM, PIXEL_SIZE_UM * row_step))
      .set_angles(theta_proj, angle_unit='degree')
      .set_labels(['angle', 'vertical', 'horizontal']))
print(ag)
show_geometry(ag);

## Load projections, flats and darks

In [ ]:
t0 = time.time()
data = ag.allocate(None)
arr = data.as_array()
for start in range(0, n_proj, FRAME_CHUNK):
    stop = min(start + FRAME_CHUNK, n_proj)
    HDF5_utilities.read_to(HDF, DSET, arr,
                           source_sel=(slice(proj_sel.start + start, proj_sel.start + stop), row_sel, slice(None)),
                           dest_sel=(slice(start, stop), slice(None), slice(None)))
    print(f'\rread {stop}/{n_proj} projections ({time.time() - t0:.0f} s)', end='')

# plain mean of all 40 flats and darks: a median is noisier (adds rings) and outlier rejection made no measurable difference
flat = HDF5_utilities.read(HDF, DSET, source_sel=(flat_sel, row_sel, slice(None))).mean(axis=0)
dark = HDF5_utilities.read(HDF, DSET, source_sel=(dark_sel, row_sel, slice(None))).mean(axis=0)
print(f'\nloaded {data.shape} in {time.time() - t0:.0f} s')

## Preprocessing: flat/dark normalisation and -log
First, before/after views of the CIL `Normaliser` (flat/dark correction) in projection space and in sinogram space. Then the -log conversion.

### Projection space
Full frames at 0° and 90° are read separately, because in slice mode `data` only holds a few detector rows.
They're normalised with the full-frame mean flat and dark.

In [ ]:
flat_full = HDF5_utilities.read(HDF, DSET, source_sel=(flat_sel, slice(None), slice(None))).mean(axis=0)
dark_full = HDF5_utilities.read(HDF, DSET, source_sel=(dark_sel, slice(None), slice(None))).mean(axis=0)

idx = [0, n_proj // 2]
ag_proj = (AcquisitionGeometry.create_Parallel3D(units='um')
           .set_panel((n_horiz, n_vert), pixel_size=PIXEL_SIZE_UM)
           .set_angles(theta_proj[idx], angle_unit='degree'))
proj_raw = ag_proj.allocate(None)
for k, i in enumerate(idx):
    HDF5_utilities.read_to(HDF, DSET, proj_raw.as_array(),
                           source_sel=(slice(proj_sel.start + i, proj_sel.start + i + 1), slice(None), slice(None)),
                           dest_sel=(slice(k, k + 1), slice(None), slice(None)))
proj_norm = Normaliser(flat_field=flat_full, dark_field=dark_full)(proj_raw)

prange = lambda a: tuple(np.percentile(a, [0.5, 99.5]))
panels = [proj_raw.get_slice(angle=0), proj_raw.get_slice(angle=1), proj_norm.get_slice(angle=0), proj_norm.get_slice(angle=1)]
show2D(panels,
       title=[f'before: raw {theta_proj[i]:.0f} deg' for i in idx] + [f'after: flat/dark corrected {theta_proj[i]:.0f} deg' for i in idx],
       fix_range=[prange(p.as_array()) for p in panels], origin='upper-left', num_cols=2, size=(16, 15));
del flat_full, dark_full, proj_raw, proj_norm, panels

### Bad-pixel masking (before normalisation)
Three kinds of outlier are looked for. All masked pixels are replaced by interpolating along `horizontal` (CIL `Masker.interpolate`),
in the raw projections **and** in the mean flat and dark, so `Normaliser` sees consistent values.
1. **Hot pixels:** mean dark above `HOT_PIXEL_DARK_COUNTS` (CIL `MaskGenerator.threshold`). Their dark current isn't stable during the scan,
   so even after dark subtraction they leave fixed errors of up to ~4%, which become sharp rings.
2. **Persistent outliers in the projections:** pixels whose angle-averaged raw counts are more than `OUTLIER_MAD` × MAD away from their
   `OUTLIER_WINDOW` horizontal neighbours (CIL `MaskGenerator.median` on the ratio to a local median). Most of these are also hot pixels.
3. **Zingers** (optional, `ZINGER_MASK`): one-off outliers in single projections, relative to a 5-projection median along the angle axis.
   Only ~4 were found in 3 rows × 4001 projections, so this is off by default (it needs a second full-size copy of the data).

*Note:* `MaskGenerator.median(..., window=...)` currently fails with SciPy ≥ 1.18. It passes the window size as `footprint=` instead of `size=` to
`scipy.ndimage.median_filter`. So the local median is computed here with SciPy, and `MaskGenerator.median` is used without `window` on the ratio.

In [ ]:
det_geom = data.geometry.get_slice(angle=0)


def det_dc(x):
    # wrap a (vertical, horizontal) array as a CIL detector-plane DataContainer
    a = det_geom.allocate(None)
    a.fill(np.asarray(x, dtype=np.float32))
    return a


good_hot = MaskGenerator.threshold(max_val=HOT_PIXEL_DARK_COUNTS)(det_dc(dark)).as_array()        # True = good
proj_mean = data.as_array().mean(axis=0)
ratio = proj_mean / median_filter(proj_mean, size=(1, OUTLIER_WINDOW), mode='reflect')
good_proj = MaskGenerator.median(axis='horizontal', threshold_factor=OUTLIER_MAD)(det_dc(ratio)).as_array()
good_2d = good_hot & good_proj
print(f'hot pixels (dark): {int((~good_hot).sum())}; persistent projection outliers: {int((~good_proj).sum())} '
      f'({int(((~good_proj) & good_hot).sum())} not hot); combined: {int((~good_2d).sum())} of {good_2d.size} ({100 * (~good_2d).mean():.2f}%)')

good_3d = np.broadcast_to(good_2d, data.shape)
if ZINGER_MASK:
    rel = data.as_array() / median_filter(data.as_array(), size=(5, 1, 1), mode='reflect')
    rel_dc = data.geometry.allocate(None)
    rel_dc.fill(rel)
    good_z = MaskGenerator.median(threshold_factor=ZINGER_MAD)(rel_dc).as_array()
    good_3d = good_3d & good_z
    print(f'zingers: {int((~good_z).sum())} values')
    del rel, rel_dc

# keep the unmasked focus row (raw counts, mean flat and dark) for the before/after comparison
sino_unmasked = data.get_slice(vertical=v_focus).copy()
flat_unmasked, dark_unmasked = flat[v_focus].copy(), dark[v_focus].copy()

if MASK_OUTLIERS:
    Masker.interpolate(mask=good_3d, axis='horizontal')(data, out=data)
    dark = Masker.interpolate(mask=good_2d, axis='horizontal')(det_dc(dark)).as_array()
    flat = Masker.interpolate(mask=good_2d, axis='horizontal')(det_dc(flat)).as_array()

In [ ]:
# before/after FBP of the focus row: flat/dark + flux normalisation, -log, edge padding, provisional COR (COR_OFFSET_PX or 0); no ring removal
ion_dark = ion[dark_sel].mean()
flux = ion[proj_sel] - ion_dark
flux_target = (ion[flat_sel] - ion_dark).mean()


def quick_fbp(raw_sino, flat_row, dark_row):
    s = Normaliser(flat_field=flat_row, dark_field=dark_row)(raw_sino)
    s = FluxNormaliser(flux=flux, target=flux_target)(s)
    s = TransmissionAbsorptionConverter()(s)
    c = s.as_array().mean(axis=0)
    fp = c - np.convolve(c, np.ones(15) / 15, 'same')        # fixed pattern: what turns into rings
    ig = s.geometry.get_ImageGeometry()
    s_pad = getattr(Padder, PAD_MODE)(pad_width={'horizontal': PAD_WIDTH_PX})(s) if PAD else s
    s_pad.geometry.set_centre_of_rotation(0.0 if COR_OFFSET_PX is None else COR_OFFSET_PX, distance_units='pixels')
    return FBP(s_pad, ig, backend=BACKEND).run(verbose=0).as_array(), fp


rec_unmasked, fp_before = quick_fbp(sino_unmasked, flat_unmasked, dark_unmasked)
rec_masked, fp_after = quick_fbp(data.get_slice(vertical=v_focus), flat[v_focus], dark[v_focus])

bad_cols = np.where(~good_2d[v_focus])[0]
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(fp_before, lw=0.5, label=f'without masking (std {fp_before[20:-20].std():.2e})')
ax.plot(fp_after, lw=0.5, label=f'with masking (std {fp_after[20:-20].std():.2e})')
ax.plot(bad_cols, fp_before[bad_cols], 'k.', ms=3, label='masked pixels')
lim = 1.2 * np.abs(fp_before[20:-20]).max()
ax.set_xlim(20, len(fp_before) - 20)
ax.set_ylim(-lim, lim)
ax.set_xlabel('horizontal (pixels)')
ax.set_title(f'fixed pattern in the -log sinogram (row {FOCUS_ROW})')
ax.legend()
plt.show()

rng = fov_range(rec_masked)
z = (slice(600, 1100), slice(600, 1100))
titles = ['FBP without masking', 'FBP with masking' if MASK_OUTLIERS else 'MASK_OUTLIERS is False']
show2D([rec_unmasked, rec_masked], title=titles, fix_range=rng, origin='upper-left', num_cols=2, size=(16, 8));
show2D([rec_unmasked[z], rec_masked[z]], title=[t + ' (centre)' for t in titles], fix_range=rng, origin='upper-left', num_cols=2, size=(16, 8));
del sino_unmasked

### Sinogram space
The focus row (`FOCUS_ROW`), before and after the in-place `Normaliser` on `data`.

In [ ]:
sino_raw = data.get_slice(vertical=v_focus).copy()
Normaliser(flat_field=flat, dark_field=dark)(data, out=data)
sino_norm = data.get_slice(vertical=v_focus).copy()

show2D([sino_raw, sino_norm], title=['before: raw counts', 'after: flat/dark corrected (transmission)'],
       fix_range=[prange(s.as_array()) for s in (sino_raw, sino_norm)], num_cols=2, size=(16, 8));
del sino_raw, sino_norm

### Flux normalisation (ion chamber)
The flats were all taken at the start of the ~4 h scan, so beam-intensity changes during the scan aren't in the flat-field correction.
CIL `FluxNormaliser` scales each projection by `target / flux[i]`. Here `flux[i]` is the ion-chamber reading for projection i and `target` is the mean reading during the flats.
Both have the ion-chamber dark offset (shutter closed) subtracted.
This corrects the overall brightness of each projection. It can't correct changes in the beam *profile*, which cause the rings.

In [ ]:
ion_dark = ion[dark_sel].mean()
flux = ion[proj_sel] - ion_dark
flux_target = (ion[flat_sel] - ion_dark).mean()
print(f'ion chamber: dark {ion_dark:.4f}, flats {flux_target + ion_dark:.4f}, projections {ion[proj_sel].min():.4f} to {ion[proj_sel].max():.4f}')

sino_before = data.get_slice(vertical=v_focus).copy()
if ION_NORMALISE:
    FluxNormaliser(flux=flux, target=flux_target)(data, out=data)
sino_after = data.get_slice(vertical=v_focus).copy()

fig, ax = plt.subplots(1, 2, figsize=(16, 4))
ax[0].plot(theta_proj, flux_target / flux, lw=0.5)
ax[0].set_xlabel('angle (deg)')
ax[0].set_title('flux scaling factor per projection (target / flux)')
ax[1].plot(theta_proj, sino_before.as_array().mean(axis=1), lw=0.5, label='without flux normalisation')
ax[1].plot(theta_proj, sino_after.as_array().mean(axis=1), lw=0.5, label='with flux normalisation')
ax[1].set_xlabel('angle (deg)')
ax[1].set_title(f'mean transmission per projection (row {FOCUS_ROW})')
ax[1].legend()
plt.show()

rng = prange(sino_after.as_array())
show2D([sino_before, sino_after], title=['without flux normalisation', 'with flux normalisation'],
       fix_range=rng, num_cols=2, size=(16, 8));
del sino_before, sino_after

### -log (absorption)

In [ ]:
TransmissionAbsorptionConverter()(data, out=data)

if MODE == 'full' and FULL_BIN > 1:
    data = Binner(roi={'vertical': (None, None, FULL_BIN), 'horizontal': (None, None, FULL_BIN)})(data)

focus = v_focus // (FULL_BIN if MODE == 'full' else 1)   # index of FOCUS_ROW in data
if MODE == 'slices':
    show2D(data, slice_list=('vertical', focus), title=f'absorption sinogram (row {FOCUS_ROW})', size=(8, 8));
else:
    show2D(data, slice_list=[('vertical', focus), ('angle', 0)],
           title=[f'absorption sinogram (row {FOCUS_ROW})', 'absorption projection 0 deg'], num_cols=2, size=(16, 6));

## 1. ROI padding
The slab is wider than the field of view, so every projection is truncated. Plain FBP then gives cupping and a bright rim.
Each sinogram row is padded horizontally with CIL `Padder` (`PAD_MODE`, `PAD_WIDTH_PX` on each side). FBP still reconstructs on the original (unpadded) image geometry `ig_roi`.
`'edge'` padding repeats the boundary values, which suits a sample that continues beyond the field of view.
Zero padding (`'constant'`) doesn't help here because it creates an artificial edge.

All later steps work on the **padded** data. In full mode at `FULL_BIN=1`, the padded projections need about 2× the memory of the unpadded ones. Use `FULL_BIN=2` if RAM is tight.

In [ ]:
b = 1 if MODE == 'slices' else FULL_BIN
zoom = tuple(slice(z.start // b, z.stop // b) for z in COR_ZOOM)


def pad(x):
    # horizontally pad an AcquisitionData for ROI reconstruction (no-op if PAD is False)
    if not PAD:
        return x
    return getattr(Padder, PAD_MODE)(pad_width={'horizontal': PAD_WIDTH_PX // b})(x)


# image geometries of the unpadded field of view: 3D for the final FBP, 2D for single slices
ig_roi = data.geometry.get_ImageGeometry()
sino_nopad = data.get_slice(vertical=focus)
ig2d = sino_nopad.geometry.get_ImageGeometry()

sino_pad = pad(sino_nopad)   # keep a reference: CIL FBP only holds a weak reference to its input
rec_nopad = FBP(sino_nopad, ig2d, backend=BACKEND).run(verbose=0).as_array()
rec_pad = FBP(sino_pad, ig2d, backend=BACKEND).run(verbose=0).as_array()

rng = fov_range(rec_pad)
show2D([rec_nopad, rec_pad], title=['without padding', f'with {PAD_MODE} padding, {PAD_WIDTH_PX} px each side' if PAD else 'PAD is False'],
       fix_range=rng, origin='upper-left', num_cols=2, size=(16, 8));

fig, ax = plt.subplots(figsize=(12, 4))
y = rec_pad.shape[0] // 2
ax.plot(rec_nopad[y], lw=0.5, label='without padding')
ax.plot(rec_pad[y], lw=0.5, label='with padding')
ax.set_ylim(rng[0] - (rng[1] - rng[0]), rng[1] + 2 * (rng[1] - rng[0]))
ax.set_xlabel('horizontal_x (pixels)')
ax.set_ylabel('attenuation (1/µm)')
ax.set_title(f'line profile through image row {y}, slice of detector row {FOCUS_ROW}')
ax.legend()
plt.show()

In [ ]:
data = pad(data)
print('padded data:', data.shape)

# provisional centre of rotation for the ring-removal sweep: COR_OFFSET_PX if set, else the CIL xcorrelation estimate
cor_est = CentreOfRotationCorrector.xcorrelation()(sino_nopad).geometry.get_centre_of_rotation(distance_units='pixels')['offset'][0]
cor = cor_est if COR_OFFSET_PX is None else COR_OFFSET_PX
print(f'xcorrelation estimate: {cor_est:.2f} px; provisional offset for ring removal: {cor:.2f} px')


def fbp_at(s, offset):
    # FBP of a padded 2D sinogram at the given COR offset, on the unpadded field-of-view geometry
    s = s.copy()
    s.geometry.set_centre_of_rotation(float(offset), distance_units='pixels')
    return FBP(s, ig2d, backend=BACKEND).run(verbose=0).as_array()

## 2. Ring removal
CIL `RingRemover` is a wavelet-FFT stripe filter applied to each sinogram. Vertical stripes in the sinogram (fixed detector errors) become rings in the slice.
- `decNum` is the number of wavelet levels. More levels remove broader stripes, which matters for the broad rings from the flat-field mismatch.
- `sigma` sets the filter strength.

Stronger settings remove more rings but can smear real low-contrast features, so check the grain boundaries and not just the rings.
Ring removal comes before the centre of rotation so the COR sweeps aren't dominated by rings. The sweep below uses the focus row (padded)
at the provisional offset. Set `RING_DECNUM`/`RING_SIGMA` (or `RING_REMOVAL = False`) and re-run from here.

In [ ]:
s_focus = data.get_slice(vertical=focus)
settings = [None] + RING_SWEEP
recs = []
for st in settings:
    s = s_focus if st is None else RingRemover(st[0], RING_WNAME, st[1], info=False)(s_focus)
    recs.append(fbp_at(s, cor))
titles = ['no ring removal' if st is None else f'decNum={st[0]}, sigma={st[1]}' for st in settings]
rng = fov_range(recs[0])

show2D([r[zoom] for r in recs], title=titles, fix_range=rng, origin='upper-left', num_cols=3, size=(21, 7 * -(-len(recs) // 3)));
show2D(recs, title=titles, fix_range=rng, origin='upper-left', num_cols=3, size=(21, 7 * -(-len(recs) // 3)));

### Apply ring removal: without and with

In [ ]:
rec_ring0 = recs[0]
if RING_REMOVAL:
    t0 = time.time()
    RingRemover(RING_DECNUM, RING_WNAME, RING_SIGMA, info=False)(data, out=data)
    print(f'ring removal (decNum={RING_DECNUM}, sigma={RING_SIGMA}) took {time.time() - t0:.0f} s')
rec_ring = fbp_at(data.get_slice(vertical=focus), cor)

rng = fov_range(rec_ring)
titles = ['without ring removal', f'with ring removal (decNum={RING_DECNUM}, sigma={RING_SIGMA})' if RING_REMOVAL else 'RING_REMOVAL is False']
show2D([rec_ring0, rec_ring], title=titles, fix_range=rng, origin='upper-left', num_cols=2, size=(16, 8));
show2D([rec_ring0[zoom], rec_ring[zoom]], title=titles, fix_range=rng, origin='upper-left', num_cols=2, size=(16, 8));

## 3. Centre of rotation
- `xcorrelation` (used above as the provisional value) is biased towards 0, because the flat-field ring pattern is roughly symmetric about the detector centre.
- **`image_sharpness`** fails on the raw data (low contrast, truncation, rings). On the bubble row it converges once the data is **flux-normalised, padded and ring-removed**,
  at full resolution with an explicit search range: about −11.4 px, stable for search ranges 20–80 px. On binned data it's unstable.
  By eye, −20 to −10 px looked best on the bubble row, which agrees.

Workflow:
1. `image_sharpness` estimate on the focus row (`FOCUS_ROW`).
2. **Coarse sweep** over `COR_COARSE_PX` and **fine sweep** around `COR_FINE_CENTRE_PX` (default: the estimate) for each selected row, to check it by eye.
3. To override, set `COR_OFFSET_PX` and re-run from the padding section.

**What to look for:** at the wrong offset, real features (the bubble boundary, small inclusions, grain boundaries) smear into arcs or double up.
`COR_ZOOM` selects the region shown, e.g. `(slice(900, 1300), slice(550, 1100))` (bubble in row 436) or `(slice(1400, 1600), slice(300, 1400))` (slab surface).

In [ ]:
s = data.get_slice(vertical=focus)
try:
    cor_sharp = CentreOfRotationCorrector.image_sharpness(backend=BACKEND, search_range=COR_SHARPNESS_SEARCH_PX // b,
                                                          tolerance=COR_SHARPNESS_TOL)(s)
    cor_sharp = cor_sharp.geometry.get_centre_of_rotation(distance_units='pixels')['offset'][0]
    print(f'image_sharpness estimate (row {FOCUS_ROW}): {cor_sharp:.2f} px')
except ValueError as e:
    cor_sharp = cor_est
    print(f'image_sharpness failed ({e}); falling back to the xcorrelation estimate {cor_est:.2f} px')

cor = cor_sharp if COR_OFFSET_PX is None else COR_OFFSET_PX
print(f'offset to apply: {cor:.2f} px' + ('' if COR_OFFSET_PX is None else ' (COR_OFFSET_PX override)'))

### Coarse sweep (focus row)

In [ ]:
s = data.get_slice(vertical=focus)
recs = [fbp_at(s, o)[zoom] for o in COR_COARSE_PX]
show2D(recs, title=[f'offset {o:g} px' for o in COR_COARSE_PX], fix_range=fov_range(recs[len(recs) // 2]),
       origin='upper-left', num_cols=4, size=(20, 5 * -(-len(recs) // 4)));

### Fine sweep (each selected row)
Set `COR_FINE_CENTRE_PX` to the best coarse offset. If the best offset differs between the top and bottom rows, the rotation axis is tilted
relative to the detector. That can be corrected with `data.geometry.set_centre_of_rotation(offset, angle=..., angle_units='degree')`.

In [ ]:
centre = cor_sharp if COR_FINE_CENTRE_PX is None else COR_FINE_CENTRE_PX
fine = centre + np.arange(-COR_FINE_HALFWIDTH_PX, COR_FINE_HALFWIDTH_PX + 1e-6, COR_FINE_STEP_PX)
n_v = data.get_dimension_size('vertical')
sweep_rows = range(n_v) if MODE == 'slices' else [n_v // 4, n_v // 2, 3 * n_v // 4]

for v in sweep_rows:
    det_row = rows[v] if MODE == 'slices' else v * b
    s = data.get_slice(vertical=v)
    recs = [fbp_at(s, o)[zoom] for o in fine]
    show2D(recs, title=[f'row {det_row}, offset {o:.1f} px' for o in fine], fix_range=fov_range(recs[len(recs) // 2]),
           origin='upper-left', num_cols=5, size=(20, 4 * -(-len(recs) // 5)));

### Apply the centre of rotation: without and with
"Without" is offset 0; "with" is the applied offset (`image_sharpness` estimate, or `COR_OFFSET_PX`).

In [ ]:
s = data.get_slice(vertical=focus)
rec_cor0 = fbp_at(s, 0.0)
data.geometry.set_centre_of_rotation(cor, distance_units='pixels')
rec_cor = fbp_at(s, cor)
print(f'using centre-of-rotation offset {cor:.2f} px')
print(data.geometry)

rng = fov_range(rec_cor)
titles = ['without COR correction (offset 0 px)', f'with COR correction (offset {cor:.2f} px)']
show2D([rec_cor0, rec_cor], title=titles, fix_range=rng, origin='upper-left', num_cols=2, size=(16, 8));
show2D([rec_cor0[zoom], rec_cor[zoom]], title=titles, fix_range=rng, origin='upper-left', num_cols=2, size=(16, 8));

## FBP reconstruction

In [ ]:
t0 = time.time()
fbp = FBP(data, ig_roi, backend=BACKEND)
if MODE == 'full':
    fbp.set_split_processing(slices_per_chunk=32)
recon = fbp.run(verbose=0)
print(f'reconstructed {recon.shape} in {time.time() - t0:.0f} s')

## Results (attenuation in 1/µm)

In [ ]:
n_z = recon.get_dimension_size('vertical')
lo, hi = fov_range(recon.as_array())
if MODE == 'slices':
    show2D([recon] * n_z, slice_list=[('vertical', i) for i in range(n_z)],
           title=[f'detector row {r}' for r in rows], fix_range=(lo, hi),
           origin='upper-left', num_cols=n_z, size=(7 * n_z, 7));
else:
    show2D(recon, slice_list=[('vertical', n_z // 2), ('horizontal_y', recon.shape[1] // 2), ('horizontal_x', recon.shape[2] // 2)],
           fix_range=(lo, hi), num_cols=3, size=(21, 7));
    show2D(recon, slice_list=[('vertical', k) for k in np.linspace(0, n_z - 1, 6, dtype=int)[1:-1]],
           fix_range=(lo, hi), origin='upper-left', num_cols=4, size=(24, 6));

In [ ]:
# histogram inside the field-of-view disc
vals = recon.as_array()[..., fov_mask(recon.shape)].ravel()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(vals, bins=300, range=tuple(np.percentile(vals, [0.1, 99.9])), color='0.3')
ax.set_xlabel('attenuation (1/µm)')
ax.set_ylabel('voxel count')
ax.set_title('Reconstructed values inside the field of view')
plt.show()

## Optional: save the reconstruction

In [ ]:
# from cil.io import TIFFWriter
# TIFFWriter(data=recon, file_name='/path/to/output/olivine_fbp/slice').write()

## Iterative reconstruction (focus row, binned)
Iterative methods on the focus row (`FOCUS_ROW`), using the fully preprocessed sinogram (flux-normalised, padded, centre of rotation applied, rings removed) and **all angles**.
For speed, the sinogram is binned horizontally by `ITER_BIN` (CIL `Binner`), so the pixels are `ITER_BIN` × larger. The FBP shown for comparison uses the same binned sinogram.

Iterative methods fit a forward model to the data. The padded columns can't be explained by an image that only covers the field of view,
so the image geometry `ig_iter` is as wide as the padded detector. The central field-of-view region is cropped out for comparison with FBP.
The data range is the width of the padding plus the detector, so `PAD_WIDTH_PX` directly sets the cost.

1. **SIRT** with non-negativity (`lower=0`).
2. **FISTA** with least squares + `TV_ALPHA` × total variation, with non-negativity enforced inside the TV proximal step (`TotalVariation(lower=0)`).
   `TV_ALPHA` depends on the data scaling (including `PIXEL_SIZE_UM`), so check it with the optional alpha sweep.

In [ ]:
from cil.plugins.tigre import ProjectionOperator
from cil.optimisation.algorithms import SIRT, FISTA
from cil.optimisation.functions import LeastSquares, TotalVariation

s_focus = data.get_slice(vertical=focus)
s_iter = Binner(roi={'horizontal': (None, None, ITER_BIN)})(s_focus) if ITER_BIN > 1 else s_focus
ig_iter = s_iter.geometry.get_ImageGeometry()
A = ProjectionOperator(ig_iter, s_iter.geometry)

# binned field-of-view geometry (for FBP) and the matching central crop of the iterative images
ig_fov = ig_iter.copy()
ig_fov.voxel_num_x, ig_fov.voxel_num_y = ig2d.voxel_num_x // ITER_BIN, ig2d.voxel_num_y // ITER_BIN
oy, ox = (ig_iter.shape[0] - ig_fov.shape[0]) // 2, (ig_iter.shape[1] - ig_fov.shape[1]) // 2
fov_crop = (slice(oy, oy + ig_fov.shape[0]), slice(ox, ox + ig_fov.shape[1]))
zoom_b = tuple(slice(z.start // ITER_BIN, z.stop // ITER_BIN) for z in zoom)

rec_fbp_iter = FBP(s_iter, ig_fov, backend=BACKEND).run(verbose=0).as_array()
print('binned sinogram:', s_iter.shape, ' iterative image geometry:', ig_iter.shape, ' field of view:', ig_fov.shape)

### SIRT with non-negativity

In [ ]:
t0 = time.time()
sirt = SIRT(initial=ig_iter.allocate(0), operator=A, data=s_iter, lower=0, update_objective_interval=10)
sirt.run(SIRT_ITERATIONS, verbose=1)
rec_sirt = sirt.solution.as_array()[fov_crop]
print(f'SIRT {SIRT_ITERATIONS} iterations in {time.time() - t0:.0f} s')

In [ ]:
plt.semilogy( sirt.objective[1:])

### FISTA: least squares + TV with non-negativity

In [ ]:
f = LeastSquares(A, s_iter)
t0 = time.time()
fista = FISTA(initial=ig_iter.allocate(0), f=f, g=TV_ALPHA * TotalVariation(max_iteration=TV_INNER_ITERATIONS, lower=0),
              update_objective_interval=10)
fista.run(FISTA_ITERATIONS, verbose=1)
rec_fista = fista.solution.as_array()[fov_crop]
print(f'FISTA-TV (alpha={TV_ALPHA}) {FISTA_ITERATIONS} iterations in {time.time() - t0:.0f} s')

### Comparison: FBP, SIRT, FISTA-TV

In [ ]:
recs = [rec_fbp_iter, rec_sirt, rec_fista]
titles = [f'FBP (bin {ITER_BIN})', f'SIRT, {SIRT_ITERATIONS} it, x ≥ 0', f'FISTA-TV, alpha={TV_ALPHA}, {FISTA_ITERATIONS} it, x ≥ 0']
rng = fov_range(rec_fbp_iter)
show2D(recs, title=titles, fix_range=rng, origin='upper-left', num_cols=3, size=(24, 8));
show2D([r[zoom_b] for r in recs], title=titles, fix_range=rng, origin='upper-left', num_cols=3, size=(24, 8));

fig, ax = plt.subplots(1, 2, figsize=(16, 4))
y = (zoom_b[0].start + zoom_b[0].stop) // 2
for r, t in zip(recs, ['FBP', 'SIRT', 'FISTA-TV']):
    ax[0].plot(r[y], lw=0.5, label=t)
ax[0].set_title(f'line profile, row {y}')
ax[0].set_xlabel('horizontal_x (pixels)')
ax[0].set_ylabel('attenuation (1/µm)')
ax[0].legend()
# SIRT and FISTA-TV minimise different objectives, so each is shown relative to its starting value
ax[1].semilogy(sirt.iterations, np.array(sirt.objective) / sirt.objective[0], label='SIRT: least squares')
ax[1].semilogy(fista.iterations, np.array(fista.objective) / fista.objective[0], label='FISTA: least squares + alpha TV')
ax[1].set_xlabel('iteration')
ax[1].set_title('objective / initial objective')
ax[1].legend()
plt.show()

### Optional: TV weight sweep
Set `TV_ALPHA_SWEEP` (e.g. `[0.3, 1, 3, 10]`) to compare FISTA-TV for several weights. Each run costs about the same as the FISTA cell above.

In [ ]:
if TV_ALPHA_SWEEP:
    recs, titles = [], []
    for a in TV_ALPHA_SWEEP:
        fa = FISTA(initial=ig_iter.allocate(0), f=f, g=a * TotalVariation(max_iteration=TV_INNER_ITERATIONS, lower=0))
        fa.run(FISTA_ITERATIONS, verbose=0)
        recs.append(fa.solution.as_array()[fov_crop][zoom_b])
        titles.append(f'alpha={a}')
    show2D(recs, title=titles, fix_range=fov_range(rec_fbp_iter), origin='upper-left',
           num_cols=min(4, len(recs)), size=(20, 5 * -(-len(recs) // 4)));